This notebook is for PCA operations on the original Wildfire dataset

# Import Lib

In [ ]:
import dataset
from tensorflow import estimator as tf_estimator
import models.losses as losses
import tensorflow as tf
from models.metrics import *
import models.cnn_autoencoder_model as cnnmodel
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.decomposition import PCA
import xgboost as xgb

# Load Dataset

In [ ]:
hparams = {
    # 数据路径
    'train_path': '../dataset/next_day_wildfire_spread_train*',
    'eval_path': '../dataset/next_day_wildfire_spread_eval*',
    'test_path': '../dataset/next_day_wildfire_spread_test*',
    
    # 特征
    'input_features': ['elevation', 'pdsi', 'NDVI', 'pr', 'sph', 'th', 'tmmn',
                  'tmmx', 'vs', 'erc', 'population', 'PrevFireMask'],
    'output_features': ['FireMask'],
    
    # 方位通道
    'azimuth_in_channel': None,
    'azimuth_out_channel': None,
    
    # 数据和模型参数
    'data_sample_size': 64,
    'sample_size': 32,
    'output_sample_size': 32,
    'batch_size': 128,
    'shuffle': False,
    'shuffle_buffer_size': 10000,
    'compression_type': None,
    'input_sequence_length': 1,
    'output_sequence_length': 1,
    'repeat': False,
    'clip_and_normalize': True,
    'clip_and_rescale': False,
    
    # 数据增强
    'random_flip': False,
    'random_rotate': False,
    'random_crop': True,
    'center_crop': False,
    
    # 其他参数
    'downsample_threshold': 0.0,
    'binarize_output': True
}

train_dataset = dataset.make_dataset(
    hparams,
    mode = tf_estimator.ModeKeys.TRAIN
)
val_dataset = dataset.make_dataset(
    hparams,
    mode = tf_estimator.ModeKeys.EVAL
)
test_dataset = dataset.make_dataset(
    hparams,
    mode = tf_estimator.ModeKeys.PREDICT
)

In [ ]:
def dataset_to_numpy(dataset):
    features_list = []
    labels_list = []
    for features, label in dataset:
        features_list.append(features.numpy())  # 提取特征并转换为 NumPy 数组
        labels_list.append(label.numpy())       # 提取标签并转换为 NumPy 数组
    return np.vstack(features_list), np.concatenate(labels_list)

# 转换数据集
train_x, train_y = dataset_to_numpy(train_dataset)
val_x, val_y = dataset_to_numpy(val_dataset)
test_x, test_y = dataset_to_numpy(test_dataset)

train_y[train_y == -1] = 0
val_y[val_y == -1] = 0
test_y[test_y == -1] = 0

In [ ]:
train_x.reshape(train_x.shape[0], -1).shape

# PCA

In [ ]:
# num_samples = train_x.shape[0]
# reshaped_train_x = train_x.reshape(num_samples, -1)
# 创建PCA实例
pca = PCA(n_components=12)

# 拟合数据并转换
pca.fit(train_x.reshape(train_x.shape[0], -1))

# 使用训练集的 PCA 参数转换所有数据集
transformed_train_x = pca.transform(train_x.reshape(train_x.shape[0], -1))
transformed_val_x = pca.transform(val_x.reshape(val_x.shape[0], -1))
transformed_test_x = pca.transform(test_x.reshape(test_x.shape[0], -1))

In [ ]:
import numpy as np

# 假设 pca.components_ 是您的PCA组件
components = pca.components_
contribution = []
# 对每个主成分进行处理
for i, component in enumerate(components):
    # 将主成分重塑为原始图像的形状
    reshaped_component = component.reshape(32, 32, 12)

    # 计算每个通道的贡献
    channel_contributions = np.sum(np.abs(reshaped_component), axis=(0, 1))

    # 归一化贡献
    normalized_contributions = channel_contributions / np.sum(channel_contributions)

    contribution.append(normalized_contributions)

    print(f"Channel Contributions to Principal Component {i+1}: {normalized_contributions}")


In [ ]:

train_y_flat = train_y.reshape(train_y.shape[0], -1)
val_y_flat = val_y.reshape(val_y.shape[0], -1)
test_y_flat = test_y.reshape(test_y.shape[0], -1)

# 假设 transformed_train_x, train_y, transformed_val_x, val_y, transformed_test_x 是你的数据集

# 创建 DMatrix
dtrain = xgb.DMatrix(transformed_train_x, label=train_y_flat)
dval = xgb.DMatrix(transformed_val_x, label=val_y_flat)
dtest = xgb.DMatrix(transformed_test_x, label=test_y_flat)

# 设置参数
params = {
    'max_depth': 3,  # 树的最大深度
    'eta': 0.1,      # 学习率
    'objective': 'binary:logistic',  # 二分类的逻辑回归问题
    'eval_metric': 'logloss'  # 评估指标
}
num_round = 100  # 训练轮数

# 训练模型
bst = xgb.train(params, dtrain, num_round, evals=[(dval, 'eval')], early_stopping_rounds=10)

# 预测
predictions = bst.predict(dtest)

# 输出预测结果
print(predictions)

In [ ]:
loaded_bst = xgb.Booster()
loaded_bst.load_model('saved_model/xgboost.model')
predictions = loaded_bst.predict(dtest)

In [ ]:
test_y_flat = test_y.reshape(test_y.shape[0], -1)

mask = test_y_flat != -1

# 应用掩码
masked_labels = test_y_flat[mask]
masked_predictions = predictions[mask]

# 计算 AUC
auc_metric = tf.keras.metrics.AUC(curve='PR')
auc_metric.update_state(masked_labels, masked_predictions)
auc = auc_metric.result().numpy()

print("AUC with masked class: %.2f" % auc)

# SHAP

In [ ]:
import shap

# 创建一个 SHAP 解释器
explainer = shap.TreeExplainer(loaded_bst)

# 计算 SHAP 值
shap_values = explainer.shap_values(dtest)

# 可视化第一个预测的 SHAP 值
shap.initjs()


In [ ]:
TITLES = [
  'Elevation',
  'Wind direction',
  'Wind velocity',
  'Min temp',
  'Max temp',
  'Humidity',
  'Precip',
  'Drought',
  'Vegetation',
  'Population density',
  'Energy release component',
  'Previous fire mask',
]

In [ ]:
np.array(shap_values).shape
average_shap_values = np.mean(shap_values, axis=0)

In [ ]:
average_shap_values = np.mean(shap_values, axis=0)

# 使用 shap.summary_plot 展示平均 SHAP 值
shap.summary_plot(average_shap_values, feature_names=TITLES, plot_type='bar')